# Simple RAG Implementation

In this notebook we will build a simple RAG application based on a structured CSV file with wine rating.

Steps:
- Load the dataset
- Encode a column using vector embedding
- Retrieve some of the rows based on a query using semantic similarity
- Generate a reply to the user's query based on the retrieved rows.

#### Visual Improvements
We will use `rich library` and `rich-theme-manager` to make the output more readable, and supress warning messages.

In [2]:
# pip install rich

In [3]:
# pip install rich-theme-manager

In [4]:
# pip install pandas

In [5]:
# pip install qdrant-client sentence-transformers

In [6]:
from rich.console import Console
from rich.style import Style
import pathlib
from rich_theme_manager import Theme, ThemeManager

THEMES = [
    Theme(
        name="dark",
        description="Dark mode theme",
        tags=["dark"],
        styles={
            "repr.own": Style(color="#e87d3e", bold=True),      # Class names
            "repr.tag_name": "dim cyan",                        # Adjust tag names 
            "repr.call": "bright_yellow",                       # Function calls and other symbols
            "repr.str": "bright_green",                         # String representation
            "repr.number": "bright_red",                        # Numbers
            "repr.none": "dim white",                           # None
            "repr.attrib_name": Style(color="#e87d3e", bold=True),    # Attribute names
            "repr.attrib_value": "bright_blue",                 # Attribute values
            "default": "bright_white on black"                  # Default text and background
        },
    ),
    Theme(
        name="light",
        description="Light mode theme",
        styles={
            "repr.own": Style(color="#22863a", bold=True),          # Class names
            "repr.tag_name": Style(color="#00bfff", bold=True),     # Adjust tag names 
            "repr.call": Style(color="#ffff00", bold=True),         # Function calls and other symbols
            "repr.str": Style(color="#008080", bold=True),          # String representation
            "repr.number": Style(color="#ff6347", bold=True),       # Numbers
            "repr.none": Style(color="#808080", bold=True),         # None
            "repr.attrib_name": Style(color="#ffff00", bold=True),  # Attribute names
            "repr.attrib_value": Style(color="#008080", bold=True), # Attribute values
            "default": Style(color="#000000", bgcolor="#ffffff"),   # Default text and background
        },
    ),
]

theme_dir = pathlib.Path("themes").expanduser()
theme_dir.expanduser().mkdir(parents=True, exist_ok=True)

theme_manager = ThemeManager(theme_dir=theme_dir, themes=THEMES)
theme_manager.list_themes()

dark = theme_manager.get("dark")
theme_manager.preview_theme(dark)

 Theme  Description       Tags  Path               
 dark   Dark mode theme   dark  themes/dark.theme  
 light  Light mode theme        themes/light.theme

                                      Theme: dark - themes/dark.theme                                      
┌───────────────────┬───────────────┬───────┬─────────┬─────────┬────────────────┬────────────────────────┐
│ style             │ color         │ color │ bgcolor │ bgcolor │ attributes     │ example                │
├───────────────────┼───────────────┼───────┼─────────┼─────────┼────────────────┼────────────────────────┤
│ default           │ bright_white  │ █████ │ black   │ █████   │ -------------- │ The quick brown fox... │
├───────────────────┼───────────────┼───────┼─────────┼─────────┼────────────────┼────────────────────────┤
│ repr.attrib_name  │ #e87d3e       │ █████ │ None    │         │ b------------- │ The quick brown fox... │
├───────────────────┼───────────────┼───────┼─────────┼─────────┼────────────────┼────────────────────────┤
│ repr.attrib_value │ bright_blue   │ █████ │ None    │         │ -------------- │ The quick brown fox... │
├───────────────────┼───────────────┼───────┼─────────┼─────────┼────────────────┼────────────────────────┤
│ repr.call         │ bright_yellow │ █████ │ None    │         │ -------------- │ The quick brown fox... │
├───────────────────┼───────────────┼───────┼─────────┼─────────┼────────────────┼────────────────────────┤
│ repr.none         │ white         │ █████ │ None    │         │ -d------------ │ The quick brown fox... │
├───────────────────┼───────────────┼───────┼─────────┼─────────┼────────────────┼────────────────────────┤
│ repr.number       │ bright_red    │ █████ │ None    │         │ -------------- │ The quick brown fox... │
├───────────────────┼───────────────┼───────┼─────────┼─────────┼────────────────┼────────────────────────┤
│ repr.own          │ #e87d3e       │ █████ │ None    │         │ b------------- │ The quick brown fox... │
├───────────────────┼───────────────┼───────┼─────────┼─────────┼────────────────┼────────────────────────┤
│ repr.str          │ bright_green  │ █████ │ None    │         │ -------------- │ The quick brown fox... │
├───────────────────┼───────────────┼───────┼─────────┼─────────┼────────────────┼────────────────────────┤
│ repr.tag_name     │ cyan          │ █████ │ None    │         │ -d------------ │ The quick brown fox... │
└───────────────────┴───────────────┴───────┴─────────┴─────────┴────────────────┴────────────────────────┘
┌─ attributes legend ──────────────────────────────────────────────────────────────────┐
│  b: bold, d: dim, i: italic, u: underline, U: double underline, B: blink, 2: blink2  │
│  r: reverse, c: conceal, s: strike, f: frame, e: encircle, o: overline, L: Link      │
└──────────────────────────────────────────────────────────────────────────────────────┘

In [7]:
from rich.console import Console

dark = theme_manager.get("dark")

# Create a console with the dark theme
console = Console(theme=dark)

In [8]:
import warnings

# Suppress warnings
warnings.filterwarnings('ignore')

## Step 1: Load the dataset

In [9]:
import pandas as pd
data = pd.read_csv("data/top_rated_wines.csv").query('variety.notna()').reset_index(drop=True).to_dict('records')
console.print(data[:2])

[
    {
        'name': '3 Rings Reserve Shiraz 2004',
        'region': 'Barossa Valley, Barossa, South Australia, Australia',
        'variety': 'Red Wine',
        'rating': 96.0,
        'notes': 'Vintage Comments : Classic Barossa vintage conditions. An average wet Spring followed by extreme 
heat in early February. Occasional rainfall events kept the vines in good balance up to harvest in late March 2004.
Very good quality coupled with good average yields. More than 30 months in wood followed by six months tank 
maturation of the blend prior to bottling, July 2007. '
    },
    {
        'name': 'Abreu Vineyards Cappella 2007',
        'region': 'Napa Valley, California',
        'variety': 'Red Wine',
        'rating': 96.0,
        'notes': 'Cappella is a proprietary blend of two clones of Cabernet Sauvignon with Cabernet Franc, Petit 
Verdot and Merlot. The gravelly soil at Cappella produces fruit that is very elegant in structure. The resulting 
wine exhibits beautiful purity of fruit with fine grained and lengthy tannins. '
    }
]

## Encode using vector embedding

We will use:
- open source vector databases: `Qdrant`  
- embedding encoder and text transformer libraries: `SentenceTransformer`


In [10]:
from qdrant_client import models, QdrantClient
from sentence_transformers import SentenceTransformer

# Create the vector database client
qdrant = QdrantClient(":memory:") # Create in memory Qdrant instance

# Create the embedding encoder
encoder = SentenceTransformer("all-MiniLM-L6-v2") # Model to create embeddings for the text data

In [11]:
# Create a collection to store the wine rating data
collection_name = "top_wines"

qdrant.recreate_collection(
    collection_name = collection_name,
    vectors_config = models.VectorParams(
        size = encoder.get_sentence_embedding_dimension(), # Vector size is defined by used model
        distance = models.Distance.COSINE # Use cosine distance for similarity search
    )
)

True

## Loading the data into the vector database
We will use the vector collection that we created above, to go over all the notes column of the wine dataset, and encode it into embedding vector, and store it in the vector database.
The indexing of the data to allow quick retrieval is running in the background as we load it.


In [12]:
qdrant.upload_points(
    collection_name = collection_name,
    points = [
        models.PointStruct(
            id = idx,
            vector = encoder.encode(doc["notes"]).tolist(),
            payload = doc
        ) for idx, doc in enumerate(data)
    ]
)

In [13]:
console.print(qdrant.get_collection(collection_name=collection_name))

CollectionInfo(
    status=<CollectionStatus.GREEN: 'green'>,
    optimizer_status=<OptimizersStatusOneOf.OK: 'ok'>,
    warnings=None,
    indexed_vectors_count=0,
    points_count=1347,
    segments_count=1,
    config=CollectionConfig(
        params=CollectionParams(
            vectors=VectorParams(
                size=384,
                distance=<Distance.COSINE: 'Cosine'>,
                hnsw_config=None,
                quantization_config=None,
                on_disk=None,
                datatype=None,
                multivector_config=None
            ),
            shard_number=None,
            sharding_method=None,
            replication_factor=None,
            write_consistency_factor=None,
            read_fan_out_factor=None,
            read_fan_out_delay_ms=None,
            on_disk_payload=None,
            sparse_vectors=None
        ),
        hnsw_config=HnswConfig(
            m=16,
            ef_construct=100,
            full_scan_threshold=10000,
            max_indexing_threads=0,
            on_disk=None,
            payload_m=None,
            inline_storage=None
        ),
        optimizer_config=OptimizersConfig(
            deleted_threshold=0.2,
            vacuum_min_vector_number=1000,
            default_segment_number=0,
            max_segment_size=None,
            memmap_threshold=None,
            indexing_threshold=20000,
            flush_interval_sec=5,
            max_optimization_threads=1,
            prevent_unoptimized=None
        ),
        wal_config=WalConfig(wal_capacity_mb=32, wal_segments_ahead=0, wal_retain_closed=1),
        quantization_config=None,
        strict_mode_config=None,
        metadata=None
    ),
    payload_schema={},
    update_queue=None
)

## Retrieve semantically relevant data based on user's query
Once the data is loaded into the vector database and the indexing process is done, we can start using our simple RAG system.

In [14]:
user_prompt = "Suggest me an amazing Chardonnay wine from Australia"

### Encoding the user's query
We will use the same encoder that we used to encode the document data to encode the query of the user. This will search results based on semantic similarity.

In [15]:
query_vector = encoder.encode(user_prompt).tolist()

### Search similar rows
We can now take the embedding encoding of the user's query and use it to find similar rows in the vector database.

In [16]:
# Search time for awesome wines
hits = qdrant.query_points(
    collection_name = collection_name,
    query = query_vector,
    limit = 5
).points

In [17]:
from rich.console import Console
from rich.text import Text
from rich.table import Table

table = Table(title="Retrieval Results", show_lines = True)

table.add_column("Name", style = "yellow")
table.add_column("Region", style = "bright_red")
table.add_column("Variety", style = "green")
table.add_column("Rating", style = "#a6accd")
table.add_column("Notes", style = "#89ddff")
table.add_column("Score", style = "violet")

for hit in hits:
    table.add_row(
        hit.payload["name"],
        hit.payload["region"],
        hit.payload["variety"],
        str(hit.payload["rating"]),
        f'{hit.payload["notes"][:50]}...',
        f'{hit.score:.4f}'
    )
    
console.print(table)

                                                 Retrieval Results                                                 
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┓
┃ Name                      ┃ Region                    ┃ Variety    ┃ Rating ┃ Notes                    ┃ Score  ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━┩
│ Greenock Creek Alice's    │ Barossa Valley, Barossa,  │ Red Wine   │ 97.0   │ Rich and fleshy, with    │ 0.6766 │
│ Shiraz 2003               │ South Australia,          │            │        │ pretty coffee, plum,     │        │
│                           │ Australia                 │            │        │ wild be...               │        │
├───────────────────────────┼───────────────────────────┼────────────┼────────┼──────────────────────────┼────────┤
│ Greenock Creek Alices     │ Barossa Valley, Barossa,  │ Red Wine   │ 96.0   │ Rich and fleshy, with    │ 0.6766 │
│ Shiraz 2002               │ South Australia,          │            │        │ pretty coffee, plum,     │        │
│                           │ Australia                 │            │        │ wild be...               │        │
├───────────────────────────┼───────────────────────────┼────────────┼────────┼──────────────────────────┼────────┤
│ Greenock Creek Alices     │ Barossa Valley, Barossa,  │ Red Wine   │ 98.0   │ Rich and fleshy, with    │ 0.6766 │
│ Shiraz 2001               │ South Australia,          │            │        │ pretty coffee, plum,     │        │
│                           │ Australia                 │            │        │ wild be...               │        │
├───────────────────────────┼───────────────────────────┼────────────┼────────┼──────────────────────────┼────────┤
│ Jean-Noel Gagnard         │ Cote de Beaune, Cote      │ White Wine │ 96.0   │ Gold-tinged and          │ 0.6338 │
│ Batard-Montrachet 2009    │ d'Or, Burgundy, France    │            │        │ extremely healthy        │        │
│                           │                           │            │        │ Chardonnay grape...      │        │
├───────────────────────────┼───────────────────────────┼────────────┼────────┼──────────────────────────┼────────┤
│ Aubert Sugar Shack Estate │ Napa Valley, California   │ White Wine │ 98.0   │ The 2017 Sugar Shack     │ 0.6242 │
│ Chardonnay 2017           │                           │            │        │ Estate Chardonnay is a   │        │
│                           │                           │            │        │ compel...                │        │
└───────────────────────────┴───────────────────────────┴────────────┴────────┴──────────────────────────┴────────┘

## Augment the prompt to the LLM with retrieved data
We will simply take the top 3 results and use them as in the prompt to the generation LLM.

## Generate reply to the user's query
We will use genAI LMM from OpenAI.

In [20]:
from dotenv import load_dotenv
load_dotenv()

True

#### First let's try without retieval
We can ask the LLM to recommend based on the user prompt.

In [25]:
# Now time to connect to the LLM
from openai import OpenAI
from rich.panel import Panel

client = OpenAI()
completion = client.chat.completions.create(
    model = "gpt-4o-mini",
    messages = [
        {
            "role": "system",
            "content": "You are a chatbot, a wine specialist. Your top priority is to help guide users to select amazing wine and guide them with their requests."
        },
        {
            "role": "user",
            "content": user_prompt
        },
        {
            "role": "assistant",
            "content": f"Here are some amazing wines that I found for you:\n\n"
        }
    ]
)

response_text = Text(completion.choices[0].message.content)

styled_panel = Panel(
    response_text,
    title="Wine Recommendation without Retrieval",
    expand=False,
    border_style="bold green",
    padding=(1, 1)
)

console.print(styled_panel)

╭───────────────────────────────────── Wine Recommendation without Retrieval ─────────────────────────────────────╮
│                                                                                                                 │
│ Absolutely! Here are a few standout Australian Chardonnays that you should consider:                            │
│                                                                                                                 │
│ 1. **Leeuwin Estate Art Series Chardonnay (Margaret River)** - This iconic Chardonnay is known for its balance  │
│ of richness and elegance. It exhibits flavors of stone fruits, citrus, and a touch of oak, complemented by a    │
│ vibrant acidity.                                                                                                │
│                                                                                                                 │
│ 2. **Elephant Hill Chardonnay (Hawke's Bay)** - This wine offers a beautifully expressive nose with aromas of   │
│ ripe citrus, stone fruits, and a hint of toasty oak. The palate is creamy and textured, with a long finish.     │
│                                                                                                                 │
│ 3. **Yerring Station Chardonnay (Yarra Valley)** - A finely structured wine with flavors of pear, apple, and a  │
│ hint of melon, it also showcases subtle oak and vibrant acidity, making it very food-friendly.                  │
│                                                                                                                 │
│ 4. **Kreglinger Vintage Brut Chardonnay (Tamar Valley)** - For something sparkling, this beautiful Chardonnay   │
│ exhibits fresh apple and pear notes with a crisp finish, perfect for celebrating special moments.               │
│                                                                                                                 │
│ 5. **Coldstream Hills Chardonnay (Yarra Valley)** - This wine has a lovely balance of ripe fruit flavors and    │
│ creamy oak influences, with notes of peach and citrus, leading to a long, satisfying finish.                    │
│                                                                                                                 │
│ Each of these Chardonnays showcases the unique terroir of their respective regions and the craftsmanship of     │
│ Australian winemakers. Enjoy your selection!                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

#### Now, add Retrieval Results
The recommendation sounds great, however we don't have this wine in our inventory and menu. Moreover, new wines may be newly avaialble that were not part of the pre-training of the LLM.

We will run the same query with the Retrieval Results and get better recommendattions for our business needs.

In [26]:
# Define a variable to hold the search results
search_results = [hit.payload for hit in hits]

In [27]:
completion_with_retrieval = client.chat.completions.create(
    model = "gpt-4o-mini",
    messages = [
        {
            "role": "system",
            "content": "You are a chatbot, a wine specialist. Your top priority is to help guide users to select amazing wine and guide them with their requests."
        },
        {
            "role": "user",
            "content": user_prompt
        },
        {
            "role": "assistant",
            "content": str(search_results)
        }
    ]
)   

response_text_with_retrieval = Text(completion_with_retrieval.choices[0].message.content)

styled_panel_with_retrieval = Panel(
    response_text_with_retrieval,
    title="Wine Recommendation with Retrieval",
    expand=False,
    border_style="bold green",
    padding=(1, 1)
)

console.print(styled_panel_with_retrieval)

╭────────────────────────────────────── Wine Recommendation with Retrieval ───────────────────────────────────────╮
│                                                                                                                 │
│ I recommend trying the **Leeuwin Estate Art Series Chardonnay** from Margaret River, Western Australia. This    │
│ wine consistently receives high praise for its elegance and complexity.                                         │
│                                                                                                                 │
│ ### Leeuwin Estate Art Series Chardonnay                                                                        │
│                                                                                                                 │
│ - **Region**: Margaret River                                                                                    │
│ - **Varietal**: Chardonnay                                                                                      │
│ - **Tasting Notes**: The wine presents a vibrant bouquet of citrus and stone fruits, with layers of oak and     │
│ subtle nuttiness. On the palate, it is rich and full-bodied, with crisp acidity and a long finish, showcasing   │
│ flavors of white peach, citrus zest, and a touch of toasty oak.                                                 │
│                                                                                                                 │
│ This Chardonnay is acclaimed for its balance and depth, making it an excellent choice for both casual sipping   │
│ and pairing with food. Enjoy it with seafood, roasted chicken, or creamy pasta dishes!                          │
│                                                                                                                 │
│ If you have any specific preferences or pairings in mind, let me know, and I can provide further                │
│ recommendations!                                                                                                │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯